# Chess Transformer Player — Step 3: Validation & Performance Test

This notebook:
1. **Validates** the submission against the official exam requirements
2. **Tests performance** in 10 games vs RandomPlayer

**Run on**: Google Colab (free tier, CPU or GPU)

No data download or retraining needed — loads model directly from HuggingFace.

In [ ]:
# ── Step 1: Install chess_exam environment ────────────────────
!git clone https://github.com/bylinina/chess_exam.git
%cd /content/chess_exam
!pip install -e . -q

In [ ]:
# ── Step 2: Official validation ───────────────────────────────
# This runs the exact same checks the professors use for disqualification
from chess_tournament import validate_player

REPO_URL = 'https://github.com/Iman-ghotbi/chess-transformer-player.git'

print(f"Validating: {REPO_URL}")
print("-" * 60)
res = validate_player(REPO_URL)
print("-" * 60)

In [ ]:
# ── Step 3: Print validation report ──────────────────────────
print("\n📋 VALIDATION REPORT")
print("=" * 40)
checks = {
    'import_ok'         : 'player.py imports without errors',
    'class_found'       : 'TransformerPlayer class exists',
    'instance_ok'       : 'Initializable with just name argument',
    'valid_move_format' : 'get_move() returns valid UCI format',
    'approved'          : 'OVERALL APPROVED',
}
all_pass = True
for key, description in checks.items():
    status = res.get(key, False)
    icon   = '✅' if status else '❌'
    print(f"{icon} {description}")
    if not status:
        all_pass = False

print(f"\nMove returned: {res.get('move_from_fen', 'N/A')}")
print(f"Load time:     {res.get('duration', 0):.1f}s")

if res.get('error_message'):
    print(f"\n⚠️  Error: {res['error_message']}")

print("\n" + ("🎉 READY TO SUBMIT!" if all_pass else "❌ FIX ERRORS BEFORE SUBMITTING"))

In [ ]:
# ── Step 4: Load player for performance testing ───────────────
# Load from the cloned repo (already downloaded by validate_player)
import importlib.util, sys

# The validate_player function clones to a folder named after the repo
# Try both possible locations
import os
possible_paths = [
    '/content/chess_exam/chess-transformer-player/player.py',
    '/content/chess-transformer-player/player.py',
    '/tmp/chess-transformer-player/player.py',
]

player_path = None
for path in possible_paths:
    if os.path.exists(path):
        player_path = path
        print(f"Found player.py at: {path}")
        break

if player_path is None:
    # Clone fresh if not found
    print("Cloning repo for testing...")
    !git clone https://github.com/Iman-ghotbi/chess-transformer-player.git /tmp/chess-transformer-player
    !pip install transformers accelerate python-chess -q
    player_path = '/tmp/chess-transformer-player/player.py'

# Load via importlib (bypasses sys.path issues)
spec   = importlib.util.spec_from_file_location("player", player_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
TransformerPlayer = module.TransformerPlayer

print("TransformerPlayer loaded ✓")

In [ ]:
# ── Step 5: Quick move quality check ─────────────────────────
import chess

my_player = TransformerPlayer("Iman-GPT2")
print("Player initialized ✓\n")

test_positions = [
    ("Starting position (white)",  "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"),
    ("After 1.e4 (black)",         "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1"),
    ("Italian Game (white)",       "r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4"),
    ("Endgame (white)",            "8/3k4/8/8/8/8/3K4/8 w - - 0 1"),
]

print("Move quality check:")
print("-" * 50)
for name, fen in test_positions:
    move  = my_player.get_move(fen)
    board = chess.Board(fen)
    legal = chess.Move.from_uci(move) in board.legal_moves if move else False
    print(f"{'✅' if legal else '❌'} {name}: {move}")

In [ ]:
# ── Step 6: 10-game performance test vs RandomPlayer ─────────
from chess_tournament import Game, RandomPlayer
import time

N_GAMES       = 10
MAX_HALF_MOVES = 150

wins, draws, losses = 0, 0, 0
total_fallbacks     = 0
game_times          = []

print(f"Running {N_GAMES} games vs RandomPlayer...")
print("-" * 60)

for i in range(N_GAMES):
    t_start = time.time()

    # Alternate colors for a fair test
    if i % 2 == 0:
        game   = Game(my_player, RandomPlayer("Random"), max_half_moves=MAX_HALF_MOVES)
        result = game.play()
        my_key = "Iman-GPT2"
    else:
        game   = Game(RandomPlayer("Random"), my_player, max_half_moves=MAX_HALF_MOVES)
        result = game.play()
        my_key = "Iman-GPT2"

    elapsed = time.time() - t_start
    game_times.append(elapsed)

    outcome  = result[0]
    score    = result[1].get(my_key, 0)
    fallback = result[2].get(my_key, 0)
    color    = "white" if i % 2 == 0 else "black"

    total_fallbacks += fallback
    if score == 1.0:   wins  += 1
    elif score == 0.5: draws += 1
    else:              losses += 1

    print(f"Game {i+1:2d} ({color:5s}): {outcome:7s} | "
          f"score: {score} | fallbacks: {fallback} | time: {elapsed:.0f}s")

print("-" * 60)

In [ ]:
# ── Step 7: Final performance report ─────────────────────────
total_score   = wins + draws * 0.5
avg_game_time = sum(game_times) / len(game_times)

print("\n" + "=" * 50)
print("         PERFORMANCE REPORT")
print("=" * 50)
print(f"  Model      : Iman-ghotbi/chess-gpt2-finetuned")
print(f"  Opponent   : RandomPlayer")
print(f"  Games      : {N_GAMES}")
print()
print(f"  Wins       : {wins}")
print(f"  Draws      : {draws}")
print(f"  Losses     : {losses}")
print(f"  Score      : {total_score}/{N_GAMES} ({total_score/N_GAMES*100:.0f}%)")
print()
print(f"  Fallbacks  : {total_fallbacks}  ← must be 0")
print(f"  Avg game   : {avg_game_time:.0f}s")
print("=" * 50)

# Verdict
if total_fallbacks == 0 and total_score >= 6:
    print("\n🎉 Excellent! Ready for the championship.")
elif total_fallbacks == 0 and total_score >= 4:
    print("\n✅ Good. Beats random, 0 fallbacks.")
elif total_fallbacks > 0:
    print("\n❌ WARNING: Fallbacks detected. Fix get_move() before submitting.")
else:
    print("\n⚠️  Weak performance. Consider retraining.")